# 03 · Data Wrangling: limpieza y creación de la variable objetivo

**Objetivo:** explorar el dataset de la API y transformar el resultado del aterrizaje (`Outcome`) en una
etiqueta binaria `Class`:

* `1` → la primera etapa **aterrizó** con éxito.
* `0` → no aterrizó (fallo o no se intentó).

| Código en `Outcome` | Significado |
|---|---|
| `True ASDS` / `False ASDS` | Aterrizaje exitoso / fallido en barcaza autónoma (*drone ship*) |
| `True RTLS` / `False RTLS` | Aterrizaje exitoso / fallido en plataforma terrestre (*Return To Launch Site*) |
| `True Ocean` / `False Ocean` | Descenso controlado exitoso / fallido sobre el océano |
| `None None` / `None ASDS` | No hubo intento de aterrizaje |

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [ ]:
import json

df = load_csv("dataset_part_1.csv", f"{IBM}/datasets/dataset_part_1.csv")
print(df.shape)
df.head()

## 1. Valores faltantes (%) y tipos de dato

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2)

In [ ]:
df.dtypes

## 2. Lanzamientos por sitio

In [ ]:
df["LaunchSite"].value_counts()

## 3. Lanzamientos por órbita

In [ ]:
df["Orbit"].value_counts()

## 4. Resultados de aterrizaje

In [ ]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes

In [ ]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

Los resultados que **no** son un aterrizaje exitoso:

In [ ]:
bad_outcomes = {outcome for outcome in landing_outcomes.keys() if not outcome.startswith("True")}
bad_outcomes

## 5. Variable objetivo `Class`

In [ ]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df["Outcome"]]
df["Class"] = landing_class
df[["Outcome", "Class"]].head(8)

In [ ]:
success_rate = df["Class"].mean()
print(f"Tasa global de aterrizajes exitosos: {success_rate:.2%}")

In [ ]:
df.to_csv(DATA_DIR / "dataset_part_2.csv", index=False)

summary = {
    "rows": int(len(df)),
    "success_rate": round(float(success_rate), 4),
    "launches_by_site": df["LaunchSite"].value_counts().to_dict(),
    "launches_by_orbit": df["Orbit"].value_counts().to_dict(),
    "landing_outcomes": {k: int(v) for k, v in landing_outcomes.items()},
    "bad_outcomes": sorted(bad_outcomes),
}
(RESULTS_DIR / "data_wrangling.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

## Conclusión

El dataset queda con la etiqueta `Class`, que es lo que intentaremos **predecir** en el notebook 07.
La tasa de éxito global sirve como línea base: un modelo útil debe superar a "predecir siempre aterriza".